# Phase 2 — Exploratory Data Analysis

**Scope note:** every chart in this notebook is computed on the **432 training-partition patients only** (one row per real patient, from `clinical_2`, filtered by `data/interim/patient_split.csv`). The 109 held-out patients are never touched here — looking at them now, even just for a chart, would let their distribution quietly influence feature-engineering or modeling decisions later.

**Why one row per patient, not all 2,029 `train_pool` rows:** `lifestyle.csv`'s jittered copies would over-represent whichever patients happen to have more repeats (up to 10x), distorting the true shape of the distributions. EDA needs the honest, unweighted population; the augmented copies are for training the model later, not for describing the population.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data_loader import load_clinical_2_full
from src.feature_mapping import harmonize
from src.config import INTERIM_DIR, TABLES_DIR, TARGET_COL, PATIENT_ID_COL
from src.viz import (
    apply_chart_style, save_fig, boxplot_by_class, grouped_bar_by_class,
    COLOR_NEGATIVE, COLOR_POSITIVE, CLASS_COLORS, CLASS_LABELS,
    SEQUENTIAL_BLUE, DIVERGING_CMAP, INK_PRIMARY, INK_SECONDARY, GRIDLINE,
)

patient_split = pd.read_csv(INTERIM_DIR / "patient_split.csv")
c2 = harmonize(load_clinical_2_full())
train_ids = set(patient_split.loc[patient_split["split"] == "train", PATIENT_ID_COL])
df = c2[c2[PATIENT_ID_COL].isin(train_ids)].reset_index(drop=True)

AGE = " Age (yrs)"
HEIGHT = "Height(Cm) "
PULSE = "Pulse rate(bpm) "

print("EDA base shape:", df.shape)
df[TARGET_COL].value_counts()

## Chart 1 — Class distribution
**Question:** How imbalanced is the target in the training partition?
**Chart type:** Bar chart of counts. **Why:** two categories, magnitude is the point — a bar chart is the least ambiguous way to show it (a pie of 2 slices would only add clutter).

In [ ]:
counts = df[TARGET_COL].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar([CLASS_LABELS[i] for i in counts.index], counts.values,
              color=[CLASS_COLORS[i] for i in counts.index], width=0.55)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 5, f"{v}\n({v/len(df):.0%})",
            ha="center", va="bottom", fontsize=9, color=INK_PRIMARY)
ax.set_ylabel("Number of patients")
ax.set_title("Class distribution — training partition (n=432 patients)")
ax.set_ylim(0, counts.max() * 1.2)
apply_chart_style(ax)
save_fig(fig, "01_class_distribution")
plt.show()

**Interpretation:** 291 patients (67%) are PCOS-negative and 141 (33%) are PCOS-positive — a moderate, not severe, imbalance (roughly 2:1). This is imbalanced enough that accuracy alone would be misleading (a model predicting "No PCOS" for everyone would score 67%), but not so extreme that resampling is mandatory — class weighting alone may be sufficient. This is decided in Phase 3/4, not here.

**Limitation:** this ratio describes the 432 training patients, not necessarily the true population prevalence of PCOS — this cohort was recruited for a PCOS study, so it is enriched relative to the general population.

## Chart 2 — Age by class
**Question:** Is patient age associated with PCOS status?
**Chart type:** Boxplot by class. **Why:** compares the full distribution (median, spread, outliers) of a continuous variable across two groups more honestly than comparing means alone.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by_class(ax, df, AGE, TARGET_COL)
ax.set_ylabel("Age (years)")
ax.set_title("Age by class")
apply_chart_style(ax)
save_fig(fig, "02_age_by_class")
plt.show()

df.groupby(TARGET_COL)[AGE].describe()

**Interpretation:** median age is 32 for PCOS-negative patients vs. 29 for PCOS-positive patients — PCOS patients in this cohort skew slightly younger, with heavy overlap between the two boxes. Age shows a real but modest association, not a sharply separating one.

**Limitation:** this is consistent with PCOS commonly presenting in the reproductive years, but a cross-sectional recruited cohort like this can't distinguish "PCOS is more common at younger ages" from "older PCOS patients in this clinic population were less likely to be recruited."

## Chart 3 — BMI by class
**Question:** Is BMI associated with PCOS status?
**Chart type:** Boxplot by class, with the WHO overweight threshold (25 kg/m²) marked for clinical context.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by_class(ax, df, "BMI", TARGET_COL)
ax.axhline(25, color=GRIDLINE, linewidth=1.2, linestyle="--", zorder=0)
ax.text(2.05, 25, "WHO overweight\nthreshold (25)", fontsize=7.5, color=INK_SECONDARY, va="center")
ax.set_ylabel("BMI (kg/m²)")
ax.set_title("BMI by class")
apply_chart_style(ax)
save_fig(fig, "03_bmi_by_class")
plt.show()

df.groupby(TARGET_COL)["BMI"].describe()

**Interpretation:** mean BMI is 23.6 (No PCOS) vs. 25.5 (PCOS) — PCOS patients trend into the overweight range on average, non-PCOS patients trend just under it. The effect is real but the boxes overlap substantially — BMI alone would not cleanly separate the classes.

**Limitation:** BMI is associated with PCOS in both directions in the literature (insulin resistance can drive weight gain, and weight gain can worsen PCOS symptoms) — this chart shows association, not which way the relationship runs.

## Chart 4 — Menstrual cycle regularity vs. PCOS
**Question:** How much more common is PCOS among patients with irregular cycles?
**Chart type:** Grouped bar of row-normalized proportions. **Why:** the question is about *rate within each cycle-regularity group*, not raw counts — normalizing makes the comparison direct.

One row (`Cycle(R/I) == 5`) was dropped here — not one of the two documented codes (2=Regular, 4=Irregular) and only a single occurrence; flagged as a likely data-entry value rather than a third category.

In [ ]:
cycle_map = {2: "Regular", 4: "Irregular"}
df_cycle = df[df["Cycle(R/I)"].isin([2, 4])].copy()
print("Rows excluded as non-standard Cycle(R/I) codes:", len(df) - len(df_cycle))
df_cycle["cycle_label"] = df_cycle["Cycle(R/I)"].map(cycle_map)
ct = pd.crosstab(df_cycle["cycle_label"], df_cycle[TARGET_COL], normalize="index") * 100
ct = ct.reindex(["Regular", "Irregular"])

fig, ax = plt.subplots(figsize=(5.5, 4))
grouped_bar_by_class(ax, ct.index.tolist(), ct[0].values, ct[1].values)
ax.set_ylabel("% of patients in that cycle-regularity group")
ax.set_title("PCOS rate by menstrual cycle regularity")
ax.set_ylim(0, 100)
apply_chart_style(ax)
save_fig(fig, "04_cycle_regularity_vs_pcos")
plt.show()

ct

**Interpretation:** among patients with a regular cycle, 21% have PCOS; among patients with an irregular cycle, 60% have PCOS — nearly a 3x difference in rate. This is the strongest categorical association found in this EDA, consistent with cycle irregularity being one of the three Rotterdam diagnostic criteria for PCOS.

**Limitation:** because cycle irregularity is part of how PCOS is diagnosed in the first place, this is expected to be a strong predictor almost by definition — it demonstrates the label is internally consistent with the diagnostic criteria, not that cycle irregularity is an independent "risk factor" discovered from the data.

## Chart 5 — Self-reported symptom prevalence by class
**Question:** How much more common are the classic hyperandrogenism/insulin-resistance symptoms among PCOS patients?
**Chart type:** Grouped bar, one panel with 5 symptoms side by side rather than 5 separate charts — the comparison is the same shape for all five, so small multiples inside one figure keeps them comparable at a glance.

In [ ]:
symptom_cols = {
    "Weight gain(Y/N)": "Weight gain",
    "hair growth(Y/N)": "Hirsutism\n(hair growth)",
    "Skin darkening (Y/N)": "Skin\ndarkening",
    "Hair loss(Y/N)": "Hair loss",
    "Pimples(Y/N)": "Acne\n(pimples)",
}
rates = pd.DataFrame({label: df.groupby(TARGET_COL)[col].mean() * 100
                       for col, label in symptom_cols.items()}).T

fig, ax = plt.subplots(figsize=(8, 4.5))
grouped_bar_by_class(ax, rates.index.tolist(), rates[0].values, rates[1].values)
ax.set_ylabel("% reporting symptom (Yes)")
ax.set_title("Self-reported symptom prevalence by class")
ax.set_ylim(0, 100)
apply_chart_style(ax)
save_fig(fig, "05_symptom_prevalence_by_class")
plt.show()

rates

**Interpretation:** every symptom is 2-3x more prevalent among PCOS patients (weight gain 21%→67%, hirsutism 13%→56%, skin darkening 15%→62%, hair loss 40%→60%, acne 39%→69%). Hirsutism and skin darkening show the sharpest jump; hair loss and acne are common even in the non-PCOS group, so they carry less discriminative power on their own.

**Limitation:** these are self-reported Yes/No symptoms, not clinically graded (e.g. no Ferriman-Gallwey hirsutism score) — reporting style and awareness of one's own diagnosis could inflate agreement in the PCOS group (recall bias).

## Chart 6 — AMH by class
**Question:** Does Anti-Müllerian Hormone (AMH) differ by PCOS status?
**Chart type:** Boxplot on a log scale — AMH is strongly right-skewed (a few very high values), so a linear scale would compress most of the distribution into a sliver at the bottom.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by_class(ax, df, "AMH(ng/mL)", TARGET_COL)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_ylabel("AMH (ng/mL), log scale")
ax.set_title("Anti-Müllerian hormone (AMH) by class")
apply_chart_style(ax)
save_fig(fig, "06_amh_by_class")
plt.show()

df.groupby(TARGET_COL)["AMH(ng/mL)"].describe()

**Interpretation:** median AMH is roughly 3.3 ng/mL (No PCOS) vs. 6.2 ng/mL (PCOS) — the PCOS group runs almost double, with a longer upper tail (max 66 vs. 26.8). A clear, clinically expected association.

**Limitation:** AMH is produced by small ovarian follicles, so it is biologically downstream of follicle count — flagged in the Phase 1 feature-mapping table as diagnostic-adjacent, not an independent risk factor. One row has missing AMH (the known stray non-numeric value, coerced to NaN during harmonization).

## Chart 7 — FSH/LH ratio by class
**Question:** Does the FSH/LH ratio, a classic PCOS marker, differ by class?
**Chart type:** Boxplot, extreme outliers hidden (a few data-entry-level extreme values would otherwise compress the box to invisibility).

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by_class(ax, df, "FSH/LH", TARGET_COL, showfliers=False)
ax.set_ylabel("FSH/LH ratio")
ax.set_title("FSH/LH ratio by class (outliers hidden for readability)")
apply_chart_style(ax)
save_fig(fig, "07_fsh_lh_ratio_by_class")
plt.show()

df.groupby(TARGET_COL)["FSH/LH"].describe()

**Interpretation:** medians are close between groups and the boxes overlap heavily — in this cohort, FSH/LH ratio does not show the sharp separation sometimes reported for PCOS, likely because of the extreme-outlier data-entry errors distorting the raw values (see Chart 9's correlation note and the outlier table below) — worth revisiting once those are cleaned in Phase 3.

**Limitation:** ratios of two noisy measurements (FSH, LH) compound their individual measurement error, and one confirmed data-entry error (FSH = 5052 mIU/mL for one patient) inflates that patient's ratio implausibly — see the outlier table.

## Chart 8 — Follicle count (left ovary) by class
**Question:** Does antral follicle count differ by class?
**Chart type:** Boxplot, with the Rotterdam ultrasound threshold (≥12 follicles) marked.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
boxplot_by_class(ax, df, "Follicle No. (L)", TARGET_COL)
ax.axhline(12, color=GRIDLINE, linewidth=1.2, linestyle="--", zorder=0)
ax.text(2.05, 12, "Rotterdam\nthreshold (12)", fontsize=7.5, color=INK_SECONDARY, va="center")
ax.set_ylabel("Left ovary follicle count")
ax.set_title("Follicle count (L) by class — ultrasound finding")
apply_chart_style(ax)
save_fig(fig, "08_follicle_count_by_class")
plt.show()

df.groupby(TARGET_COL)["Follicle No. (L)"].describe()

**Interpretation:** this is the single sharpest separator found in the whole EDA — mean follicle count 4.3 (No PCOS) vs. 9.8 (PCOS), with the PCOS group's median sitting almost exactly at the clinical threshold (12) that non-PCOS patients rarely reach.

**Limitation — important:** follicle count on ultrasound is one of the three Rotterdam diagnostic criteria for PCOS. This chart is close to showing "the label agrees with the criteria used to assign the label," not an independently discovered risk pattern. As flagged in the Phase 1 feature-mapping table, this feature (and follicle-right, average follicle size) should be treated as diagnostic-adjacent — expect a model using it to score very well, and report a sensitivity model without it to see how much signal survives from genuinely independent variables.

## Chart 9 — Correlation heatmap among numeric predictors
**Question:** Which numeric features move together, independent of the target?
**Chart type:** Diverging heatmap (blue=negative, red=positive, white=none). **Why:** this is a multicollinearity check, not a target-association check — several of these features are shown against the target already in Charts 2-8, so repeating that here would be redundant.

In [ ]:
corr_cols = [
    AGE, "Weight (Kg)", HEIGHT, "BMI", PULSE, "RR (breaths/min)", "Hb(g/dl)",
    "Cycle length(days)", "FSH(mIU/mL)", "LH(mIU/mL)", "FSH/LH", "Hip(inch)",
    "Waist(inch)", "Waist:Hip Ratio", "TSH (mIU/L)", "AMH(ng/mL)", "PRL(ng/mL)",
    "Vit D3 (ng/mL)", "RBS(mg/dl)", "BP _Systolic (mmHg)", "BP _Diastolic (mmHg)",
    "Follicle No. (L)", "Follicle No. (R)", "Avg. F size (L) (mm)", "Avg. F size (R) (mm)",
    "Endometrium (mm)",
]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(corr.values, cmap=DIVERGING_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
short_labels = [c.strip() for c in corr_cols]
ax.set_xticklabels(short_labels, rotation=90, fontsize=7.5, color=INK_SECONDARY)
ax.set_yticklabels(short_labels, fontsize=7.5, color=INK_SECONDARY)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.ax.tick_params(labelsize=8, colors=INK_SECONDARY)
cbar.set_label("Pearson correlation", color=INK_SECONDARY, fontsize=8.5)
ax.set_title("Correlation among numeric features (training partition)", color=INK_PRIMARY, pad=12)
for spine in ax.spines.values():
    spine.set_visible(False)
save_fig(fig, "09_correlation_heatmap")
plt.show()

corr_abs = corr.abs()
pairs = (corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool)).stack().sort_values(ascending=False))
pairs.head(10)

**Interpretation:** the strongest pairs are FSH vs. FSH/LH (0.97 — expected, one is derived from the other), Weight vs. BMI (0.90 — expected, BMI is derived from weight/height), Hip vs. Waist (0.87), and Follicle-L vs. Follicle-R (0.80, the same anatomical measurement on both ovaries). None of this is surprising, but it matters for Phase 4: a linear model (logistic regression) would need to watch for multicollinearity among these derived/redundant groups, while tree-based models (Random Forest, Gradient Boosting) are largely insensitive to it.

**Limitation:** Pearson correlation only captures linear relationships — a feature could still be a strong nonlinear predictor without showing up here.

## Chart 10 — Missing data overview
**Question:** How much and where is data missing?
**Chart type:** Horizontal bar of missing-value counts per column.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.barh([c.strip() for c in missing.index], missing.values, color=SEQUENTIAL_BLUE[3])
for b, v in zip(bars, missing.values):
    ax.text(v + 0.03, b.get_y() + b.get_height()/2, str(v), va="center", fontsize=9, color=INK_PRIMARY)
ax.set_xlabel(f"Missing values (out of {len(df)} patients)")
ax.set_title("Missing data by column — training partition")
ax.invert_yaxis()
apply_chart_style(ax)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(False)
save_fig(fig, "10_missing_data_overview")
plt.show()

missing

**Interpretation:** missingness is minimal — 4 columns have exactly 1 missing value each, out of 432 patients (<0.3% each). This is a high-quality, mostly-complete dataset; imputation strategy in Phase 3 will barely matter for the final result given how few values are affected.

**Limitation:** this describes only structurally missing (NaN) values. The next section shows values that are *present* but implausible — a different and, in this dataset, larger data-quality issue than missingness itself.

## Data-entry errors and outliers found in this EDA

Not one of the 8-12 "required" charts, but a real finding that surfaced while building them and needs to carry into Phase 3.

In [ ]:
checks = [
    ("Vit D3 (ng/mL)", lambda s: s > 200, "implausibly high (normal range ~20-100 ng/mL)"),
    ("Pulse rate(bpm) ", lambda s: s < 40, "implausibly low for a living patient"),
    ("FSH(mIU/mL)", lambda s: s > 100, "implausibly high (likely misplaced decimal/extra digit)"),
    ("BP _Systolic (mmHg)", lambda s: s < 50, "implausibly low (likely missing a digit)"),
    ("BP _Diastolic (mmHg)", lambda s: s < 30, "implausibly low (likely missing a digit)"),
]
rows = []
for col, cond, note in checks:
    flagged = df[cond(df[col])]
    for _, r in flagged.iterrows():
        rows.append({"Patient File No.": r[PATIENT_ID_COL], "column": col, "value": r[col],
                      "PCOS (Y/N)": r[TARGET_COL], "likely_issue": note})

outliers = pd.DataFrame(rows)
outliers.to_csv(TABLES_DIR / "outlier_flags_train_partition.csv", index=False)
print(f"{len(outliers)} flagged values across {outliers['Patient File No.'].nunique()} distinct patients")
outliers

**Interpretation:** 7 physiologically impossible values across 7 distinct patients (1.6% of the training partition) — a Vit D3 reading of 6014.66 ng/mL, a pulse rate of 13 bpm, an FSH of 5052 mIU/mL, and blood-pressure readings of 12 and 8 mmHg. Every one of these looks like a data-entry error (an extra/misplaced digit) rather than a real measurement. Each affected patient has exactly one bad value — the rest of their record looks normal.

**Decision for Phase 3:** these should be treated as missing (not deleted wholesale — the patient's other 43 features are fine) and imputed the same way as genuine missing values, or capped at a domain-informed plausible range. This is a preprocessing decision, not made here — this notebook only surfaces and records the issue.

## Chart 11 — Additional numeric variables by class (Waist:Hip ratio, Vitamin D3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, col, title in zip(axes, ["Waist:Hip Ratio", "Vit D3 (ng/mL)"],
                           ["Waist:Hip ratio by class", "Vitamin D3 by class"]):
    boxplot_by_class(ax, df, col, TARGET_COL, showfliers=False)
    ax.set_title(title, fontsize=10)
    apply_chart_style(ax)
save_fig(fig, "11_additional_numeric_by_class")
plt.show()

print(df.groupby(TARGET_COL)["Waist:Hip Ratio"].describe())
print()
print(df.groupby(TARGET_COL)["Vit D3 (ng/mL)"].describe())

**Interpretation:** Waist:Hip ratio is nearly identical between classes (0.889 vs 0.893 mean) — no meaningful association in this cohort, despite waist:hip ratio being a common metabolic-risk marker elsewhere. Vitamin D3's mean is wildly distorted by the two extreme outlier values found above (6014.66, 5418.60, both in the PCOS group) — the median (25.7 vs 27.2) tells the real story: no meaningful difference by class, once the two data-entry errors are set aside.

**Limitation:** this is a clear example of why means alone are misleading with dirty data — always check the median and the boxplot, not just a summary mean, especially before writing an interpretation.

## Chart 12 — BMI vs. follicle count, jointly, by class
**Question:** Do the two strongest univariate signals (Chart 3, Chart 8) separate the classes even better together?
**Chart type:** Scatter plot colored by class — the standard form for a bivariate relationship between two continuous variables.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for c in [0, 1]:
    sub = df[df[TARGET_COL] == c]
    ax.scatter(sub["BMI"], sub["Follicle No. (L)"], s=22, alpha=0.65,
               color=CLASS_COLORS[c], label=CLASS_LABELS[c], edgecolors="none")
ax.set_xlabel("BMI (kg/m²)")
ax.set_ylabel("Follicle No. (L)")
ax.set_title("BMI vs. follicle count, by class")
ax.legend(frameon=False, fontsize=9)
apply_chart_style(ax)
save_fig(fig, "12_bmi_vs_follicle_scatter")
plt.show()

**Interpretation:** separation runs almost entirely along the vertical (follicle count) axis — orange (PCOS) points sit visibly higher regardless of BMI. BMI spreads both colors widely along the horizontal axis with heavy overlap. This confirms what Charts 3 and 8 showed separately: follicle count is doing most of the separating work here, BMI is a weaker, noisier signal on its own.

**Limitation:** a 2D view of 2 features can't show how the *other* 40 features might interact with these two — this is a sanity check on the two strongest univariate signals, not a substitute for feature importance analysis in Phase 4/8.

## Summary

- **Strongest associations with PCOS:** follicle count (Chart 8) and cycle irregularity (Chart 4) — both are Rotterdam diagnostic-criterion-adjacent, so expected to dominate any model; report a sensitivity model without them to see what survives from genuinely independent variables.
- **Moderate, clinically expected associations:** AMH (Chart 6), all five self-reported symptoms (Chart 5), BMI (Chart 3).
- **Weak or no association found:** Age (Chart 2, modest), Waist:Hip ratio (Chart 11, essentially none), FSH/LH ratio (Chart 7, obscured by outliers).
- **Data quality:** missingness is minimal (Chart 10) but 7 patients (1.6%) carry a physiologically impossible value in one field each — flagged in `reports/tables/outlier_flags_train_partition.csv` for handling in Phase 3.
- **Multicollinearity to watch in Phase 4:** FSH/FSH-LH-ratio, Weight/BMI, Hip/Waist, Follicle-L/Follicle-R (Chart 9) — matters for logistic regression, not for tree-based models.